Task 01 | Nested Transaction Analyzer

In [1]:
from collections import defaultdict

def build_report(transactions):
    # 1. Reject duplicate IDs (checked across ALL transactions, failed ones included)
    seen = set()
    for t in transactions:
        if t["id"] in seen:
            raise ValueError(f"Duplicate transaction id: {t['id']}")
        seen.add(t["id"])

    # 2. Every customer appears, even if they have no successful transactions
    spend = {c: defaultdict(int) for c in {t["customer"] for t in transactions}}
    for t in transactions:
        if t["status"] == "success":
            spend[t["customer"]][t["category"]] += t["amount"]

    # 3. Build rows: total and top category (ties broken alphabetically)
    rows = [
        {
            "customer": c,
            "total": sum(cats.values()),
            "top_category": min(cats, key=lambda k: (-cats[k], k)) if cats else None,
        }
        for c, cats in spend.items()
    ]

    # 4. Sort by spending desc, then name asc
    rows.sort(key=lambda r: (-r["total"], r["customer"]))

    # 5. Rank: equal totals share a rank (1, 2, 2, 4 style)
    for i, r in enumerate(rows):
        r["rank"] = rows[i - 1]["rank"] if i and r["total"] == rows[i - 1]["total"] else i + 1

    return rows